# Sensor Repair — Notebook 03: Learn when to intervene

**All executable code is in one cell. Use a CPU runtime.** The notebook reads Notebook 02 action caches; it does not retrain the CNNs or require a GPU.

1. Keep Notebook 02 outputs at `My Drive/SensorRepair_ICDSIS/outputs/notebook02_v1`.
2. Open this notebook in Colab, choose a CPU runtime, run the single code cell, and authorize Drive access.
3. If interrupted, rerun the same cell with the same settings. Completed benefit/harm heads are reused.
4. Share the final output beginning `NOTEBOOK 03 COMPLETE` and these files from `My Drive/SensorRepair_ICDSIS/outputs/notebook03_v1`:
   - `notebook03_pilot_summary.csv`
   - `notebook03_run_report.json`
   - `notebook03_selected_policies.json`

The method fits two small boosted-tree models using the **policy-training partition only**. One predicts whether an action will correct a wrong prediction; the other predicts whether it will break a correct prediction. Actions are scored by estimated benefit minus a penalty times estimated harm. The original input is kept unless a candidate action exceeds the selected threshold.

Features include class probabilities, prediction disagreement, reconstruction residuals, signal statistics and the identity of the candidate action. Ground-truth labels supply training targets. Participant IDs, window IDs, fault type/severity and injected fault location are excluded from features.

Validation data selects thresholds and the harm penalty from fixed grids. The empirical selection conditions require:
- Clean Macro-F1 at least keep-baseline clean Macro-F1 minus **0.5 percentage points**.
- Harmful interventions at most **1%** of clean validation cases.
- Mean harmful interventions at most **1%** across the nine corrupted validation conditions.

A keep-only setting is always available. These are checks on the observed validation data; final evaluation must assess performance on new participants and faults. Results on this same selection set are exploratory and do not establish statistical significance or a deployment risk guarantee.

The notebook also tunes confidence-margin and residual baselines under the same conditions, includes a benefit-only ablation, and verifies all original Notebook 02 baseline metrics before fitting the policy. It reports comparison against the strongest non-oracle baseline, including the original unrestricted baselines.

Both datasets are trained separately. This pilot uses **fold_0 and seed 2026**. Final test predictions and the held-out stuck fault remain reserved. All four DSADS folds and repeated model seeds are required for the final study.

Implementation was checked with synthetic cache fixtures, independent metric parity checks, feature/target alignment checks, validation-condition checks and saved-model reuse. The actual learned-policy results require your run using your existing caches.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import hashlib
import json
import os
import shutil
import time
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from threadpoolctl import threadpool_limits

PROJECT = Path('/content/drive/MyDrive/SensorRepair_ICDSIS')
SOURCE = PROJECT / 'outputs' / 'notebook02_v1'
OUTPUT = PROJECT / 'outputs' / 'notebook03_v1'
LOCAL = Path('/content/sensorrepair_notebook03_v1')
SEED = 2026
CPU_THREADS = 2
MAX_ITER = 160
CODE_VERSION = 'sensorrepair_nb03_v1'
CELL_SIGNATURE = '6ea2f258f86b82bc97b59e2ea4b0a95f0dda8ffb1c5dcc3f3e17e07014e0b96c'
CLEAN_F1_TOLERANCE = 0.005
MAX_CLEAN_HARM = 0.01
MAX_CORRUPTED_HARM = 0.01
UTILITY_THRESHOLDS = [0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 2.0]
HARM_PENALTIES = [1.0, 2.0, 4.0]
CONFIDENCE_MARGINS = [0.0, 0.01, 0.02, 0.05, 0.1, 0.2, 2.0]
RESIDUAL_THRESHOLDS = [0.0, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0, 128.0, None]
OUTPUT.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)
print('NOTEBOOK 03 | learned benefit-and-harm policy | CPU | one executable cell', flush=True)
print('Scope: both datasets, fold_0, seed 2026; validation selection only.')


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def json_signature(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)


def save_csv(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    value.to_csv(temporary, index=False)
    os.replace(temporary, path)


def save_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temporary, path)


def atomic_joblib(path, value):
    temporary = LOCAL / (Path(path).name + '.tmp')
    joblib.dump(value, temporary, compress=3)
    partial = Path(path).with_name(Path(path).name + '.partial')
    shutil.copyfile(temporary, partial)
    os.replace(partial, path)


for filename in ['notebook02_run_report.json', 'notebook02_artifact_inventory.json']:
    if not (SOURCE / filename).is_file():
        raise FileNotFoundError(f'Missing {SOURCE / filename}. Finish Notebook 02 first.')
source_report = json.loads((SOURCE / 'notebook02_run_report.json').read_text())
assert source_report['datasets_completed'] == ['dsads', 'har']
assert source_report['fold'] == 'fold_0' and source_report['seed'] == SEED
assert source_report['test_predictions_computed'] == 0
assert source_report['held_out_faults_evaluated'] == []
source_inventory = {entry['filename']: entry for entry in json.loads((SOURCE / 'notebook02_artifact_inventory.json').read_text())}


def source_file(relative):
    if relative not in source_inventory:
        raise KeyError(f'Notebook 02 inventory does not contain {relative}. Rerun its final summary if needed.')
    expected = source_inventory[relative]['sha256']
    target = LOCAL / 'inputs' / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and file_sha256(target) == expected:
        return target
    temporary = target.with_name(target.name + '.partial')
    shutil.copyfile(SOURCE / relative, temporary)
    assert file_sha256(temporary) == expected, f'Input hash mismatch: {relative}'
    os.replace(temporary, target)
    return target


def load_pack(dataset, split):
    assert split in ['policy', 'validation']
    run_name = f'{dataset}_fold_0_seed_{SEED}'
    done = json.loads(source_file(run_name + '/dataset_complete.json').read_text())
    conditions = ['clean'] + [f'{fault}_{level}' for fault in ['dropout', 'bias', 'gain'] for level in [1, 2, 3]]
    matrices, residuals, statistics, fingerprints = [], [], [], []
    anchor = None
    for condition in conditions:
        relative = run_name + f'/action_cache/{split}__{condition}.npz'
        with np.load(source_file(relative), allow_pickle=False) as cached:
            assert cached['signature'].item() == done['cache_signature']
            assert cached['condition'].item() == condition and cached['split'].item() == split
            identity = {key: cached[key] for key in ['y', 'subjects', 'window_ids', 'source_indices', 'action_names']}
            if anchor is None:
                anchor = identity
            else:
                for key in identity:
                    assert np.array_equal(identity[key], anchor[key]), f'Case alignment mismatch: {dataset}/{split}/{key}'
            matrices.append(cached['probabilities'])
            residuals.append(cached['reconstruction_residuals'])
            statistics.append(cached['observed_group_statistics'])
        fingerprints.append({'filename': relative, 'sha256': source_inventory[relative]['sha256']})
    pack = dict(anchor)
    pack.update(probabilities=np.stack(matrices), residuals=np.stack(residuals),
                statistics=np.stack(statistics), conditions=conditions, fingerprints=fingerprints)
    p, r, s = pack['probabilities'], pack['residuals'], pack['statistics']
    D, N, A, K = p.shape
    G = r.shape[-1]
    assert D == 10 and A == 1 + 2 * G and r.shape == (D, N, G) and s.shape == (D, N, G, 4)
    assert np.isfinite(p).all() and np.isfinite(r).all() and np.isfinite(s).all()
    assert np.all(p >= 0) and np.allclose(p.sum(axis=-1), 1.0, atol=1e-5)
    assert np.all(r >= 0) and np.array_equal(np.unique(pack['y']), np.arange(K))
    assert len(np.unique(pack['window_ids'])) == N
    names = pack['action_names'].tolist()
    assert names[0] == 'keep' and all(n.startswith('mask:') for n in names[1:1 + G])
    assert all(n.startswith('reconstruct:') for n in names[1 + G:])
    assert [n.split(':', 1)[1] for n in names[1:1 + G]] == [n.split(':', 1)[1] for n in names[1 + G:]]
    return pack


def probability_statistics(probability):
    sorted_p = np.sort(probability, axis=-1)
    confidence = sorted_p[..., -1]
    margin = confidence - sorted_p[..., -2]
    entropy = -(probability * np.log(np.clip(probability, 1e-7, 1.0))).sum(axis=-1)
    return np.stack([confidence, margin, entropy], axis=-1)


def policy_features(probabilities, residuals, statistics, residual_scale):
    # This function accepts observable arrays only. It cannot access labels, subjects,
    # source IDs, injected fault locations, or condition/severity metadata.
    D, N, A, K = probabilities.shape
    G, J, M = residuals.shape[-1], A - 1, D * N
    p = probabilities.reshape(M, A, K)
    r = residuals.reshape(M, G)
    s = statistics.reshape(M, G, 4)
    original, candidates = p[:, 0], p[:, 1:]
    candidate_groups = np.tile(np.arange(G), 2)
    original_statistics = probability_statistics(original)
    candidate_statistics = probability_statistics(candidates)
    scaled = r / residual_scale[None, :]

    def repeat(values):
        return np.broadcast_to(values[:, None, :], (M, J, values.shape[-1]))

    middle = (original[:, None, :] + candidates) / 2.0
    js = 0.5 * (original[:, None, :] * np.log(np.clip(original[:, None, :], 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    js += 0.5 * (candidates * np.log(np.clip(candidates, 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    names, pieces = [], []

    def add(values, labels):
        assert values.shape == (M, J, len(labels))
        pieces.append(values)
        names.extend(labels)

    add(repeat(original), [f'original_probability_{k}' for k in range(K)])
    add(candidates, [f'action_probability_{k}' for k in range(K)])
    add(candidates - original[:, None, :], [f'probability_difference_{k}' for k in range(K)])
    add(repeat(original_statistics), ['original_confidence', 'original_margin', 'original_entropy'])
    add(candidate_statistics, ['action_confidence', 'action_margin', 'action_entropy'])
    add(candidate_statistics - original_statistics[:, None, :], ['confidence_difference', 'margin_difference', 'entropy_difference'])
    add(js[..., None], ['prediction_js_divergence'])
    add((candidates.argmax(axis=-1) == original.argmax(axis=-1)[:, None])[..., None], ['prediction_agreement'])
    kinds = np.zeros((J, 2), dtype=np.float32)
    kinds[:G, 0], kinds[G:, 1] = 1.0, 1.0
    add(np.broadcast_to(kinds[None], (M, J, 2)), ['action_is_mask', 'action_is_reconstruction'])
    add(np.broadcast_to(np.eye(G, dtype=np.float32)[candidate_groups][None], (M, J, G)), [f'candidate_group_{g}' for g in range(G)])
    add(repeat(np.log1p(r)), [f'log_residual_group_{g}' for g in range(G)])
    add(np.log1p(r[:, candidate_groups])[..., None], ['candidate_log_residual'])
    add(np.log1p(scaled[:, candidate_groups])[..., None], ['candidate_log_scaled_residual'])
    add(repeat(np.log1p(scaled.max(axis=-1, keepdims=True))), ['maximum_log_scaled_residual'])
    add(s[:, candidate_groups, :], ['candidate_mean_absolute_signal', 'candidate_signal_std', 'candidate_temporal_difference', 'candidate_zero_fraction'])
    add(repeat(s.reshape(M, G * 4)), [f'observed_group_{g}_stat_{j}' for g in range(G) for j in range(4)])
    values = np.ascontiguousarray(np.concatenate(pieces, axis=-1).reshape(M * J, -1), dtype=np.float32)
    assert len(set(names)) == len(names) and np.isfinite(values).all()
    return values, names


def outcome_targets(pack):
    predictions = pack['probabilities'].argmax(axis=-1)
    correct = predictions == pack['y'][None, :, None]
    original = correct[:, :, 0:1]
    fix = (~original & correct[:, :, 1:]).reshape(-1).astype(np.int8)
    harm = (original & ~correct[:, :, 1:]).reshape(-1).astype(np.int8)
    assert not np.any(fix & harm)
    return fix, harm


def fit_head(name, X, targets, run, signature, feature_names):
    path = run / f'{name}_head.joblib'
    if path.exists():
        payload = joblib.load(path)
        assert payload['signature'] == signature and payload['feature_names'] == feature_names, 'Stale policy model.'
        print(f'  {name} head: reused saved model.', flush=True)
        return payload['model'], payload['training_seconds']
    started = time.perf_counter()
    if len(np.unique(targets)) == 1:
        model = {'constant_probability': float(targets[0])}
    else:
        model = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=MAX_ITER,
                max_leaf_nodes=15, max_depth=4, min_samples_leaf=40, l2_regularization=5.0,
                max_bins=64, categorical_features=None, early_stopping=False,
                random_state=SEED + (1 if name == 'harm' else 0))
        model.fit(X, targets)
    seconds = time.perf_counter() - started
    atomic_joblib(path, {'signature': signature, 'model': model, 'feature_names': feature_names,
                        'training_seconds': seconds, 'positive_targets': int(targets.sum()), 'training_rows': len(targets)})
    print(f'  {name} head fitted | positive outcomes {int(targets.sum())}/{len(targets)} | {seconds:.1f}s', flush=True)
    return model, seconds


def predict_head(model, X):
    if isinstance(model, dict):
        return np.full(len(X), model['constant_probability'], dtype=np.float32)
    indices = np.flatnonzero(model.classes_ == 1)
    return model.predict_proba(X)[:, int(indices[0])].astype(np.float32) if len(indices) else np.zeros(len(X), dtype=np.float32)


def macro_f1(y, prediction, classes):
    matrix = np.bincount(y * classes + prediction, minlength=classes * classes).reshape(classes, classes)
    numerator = 2.0 * matrix.diagonal()
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    return float(np.divide(numerator, denominator, out=np.zeros(classes), where=denominator > 0).mean())


def evaluate_choices(pack, choices, method):
    p, y = pack['probabilities'], pack['y']
    D, N, A, K = p.shape
    assert choices.shape == (D, N) and choices.min() >= 0 and choices.max() < A
    predictions = p.argmax(axis=-1)
    original_correct = predictions[:, :, 0] == y[None, :]
    selected = np.take_along_axis(predictions, choices[:, :, None], axis=2).squeeze(2)
    rows = []
    for i, condition in enumerate(pack['conditions']):
        correct = selected[i] == y
        helpful = float((~original_correct[i] & correct).mean())
        harmful = float((original_correct[i] & ~correct).mean())
        accuracy = float(correct.mean())
        assert abs(accuracy - original_correct[i].mean() - helpful + harmful) < 1e-10
        rows.append({'condition': condition, 'method': method, 'windows': N,
                     'macro_f1': macro_f1(y, selected[i], K), 'accuracy': accuracy,
                     'intervention_rate': float((choices[i] != 0).mean()), 'helpful_rate': helpful,
                     'harmful_rate': harmful,
                     'harm_given_original_correct': float(np.sum(original_correct[i] & ~correct) / max(1, original_correct[i].sum()))})
    return rows


def aggregate_rows(rows):
    clean = next(row for row in rows if row['condition'] == 'clean')
    corrupted = [row for row in rows if row['condition'] != 'clean']
    answer = {key: float(np.mean([row[key] for row in corrupted])) for key in
              ['macro_f1', 'accuracy', 'intervention_rate', 'helpful_rate', 'harmful_rate']}
    answer.update(clean_macro_f1=clean['macro_f1'], clean_accuracy=clean['accuracy'],
                  clean_harmful_rate=clean['harmful_rate'], clean_intervention_rate=clean['intervention_rate'])
    return answer


def baseline_choices(pack, scale):
    p, residual = pack['probabilities'], pack['residuals']
    D, N, A, K = p.shape
    G = residual.shape[-1]
    ratio = residual / scale[None, None, :]
    suspect = ratio.argmax(axis=-1)
    gate = ratio.max(axis=-1) > 1.0
    original_correct = p.argmax(axis=-1) == pack['y'][None, :, None]
    return {
        'keep': np.zeros((D, N), dtype=np.int64),
        'mask_suspect': 1 + suspect,
        'reconstruct_suspect': 1 + G + suspect,
        'residual_gate_mask': np.where(gate, 1 + suspect, 0),
        'residual_gate_reconstruct': np.where(gate, 1 + G + suspect, 0),
        'confidence_select': p.max(axis=-1).argmax(axis=-1),
        'oracle_diagnostic': original_correct.argmax(axis=-1)
    }


def choose_actions(pack, probability_fix, probability_harm, scale, settings):
    family = settings['family']
    D, N, A, K = pack['probabilities'].shape
    if family in ['harm_aware_policy', 'benefit_only_policy']:
        scores = probability_fix - settings['harm_penalty'] * probability_harm
        selected = scores.argmax(axis=-1)
        return np.where(scores.max(axis=-1) > settings['threshold'], selected + 1, 0)
    if family == 'confidence_margin_tuned':
        confidence = pack['probabilities'].max(axis=-1)
        selected = confidence.argmax(axis=-1)
        return np.where(confidence.max(axis=-1) > confidence[:, :, 0] + settings['threshold'], selected, 0)
    if settings['threshold'] is None:
        return np.zeros((D, N), dtype=np.int64)
    ratio = pack['residuals'] / scale[None, None, :]
    selected = ratio.argmax(axis=-1)
    offset = 1 if family == 'residual_mask_tuned' else 1 + ratio.shape[-1]
    return np.where(ratio.max(axis=-1) > settings['threshold'], selected + offset, 0)


def candidates_for(family):
    if family == 'harm_aware_policy':
        return [{'family': family, 'harm_penalty': penalty, 'threshold': threshold}
                for penalty in HARM_PENALTIES for threshold in UTILITY_THRESHOLDS]
    if family == 'benefit_only_policy':
        return [{'family': family, 'harm_penalty': 0.0, 'threshold': threshold} for threshold in UTILITY_THRESHOLDS]
    thresholds = CONFIDENCE_MARGINS if family == 'confidence_margin_tuned' else RESIDUAL_THRESHOLDS
    return [{'family': family, 'threshold': threshold} for threshold in thresholds]


def select_settings(pack, probability_fix, probability_harm, scale, family, clean_keep_f1):
    best, trials = None, []
    for number, settings in enumerate(candidates_for(family)):
        actions = choose_actions(pack, probability_fix, probability_harm, scale, settings)
        summary = aggregate_rows(evaluate_choices(pack, actions, family))
        feasible = (summary['clean_macro_f1'] >= clean_keep_f1 - CLEAN_F1_TOLERANCE - 1e-12
                    and summary['clean_harmful_rate'] <= MAX_CLEAN_HARM + 1e-12
                    and summary['harmful_rate'] <= MAX_CORRUPTED_HARM + 1e-12)
        record = {'trial': number, 'family': family, 'settings_json': json.dumps(settings),
                  'feasible': bool(feasible), **summary}
        trials.append(record)
        rank = (summary['macro_f1'], summary['clean_macro_f1'], -summary['harmful_rate'], -summary['intervention_rate'])
        if feasible and (best is None or rank > best['rank']):
            best = {'rank': rank, 'settings': settings, 'actions': actions, 'summary': summary}
    assert best is not None, 'The explicit keep-only candidate must be feasible.'
    return best, trials


def verify_baseline_parity(dataset, rows):
    expected = pd.read_csv(source_file('notebook02_validation_comparison.csv'))
    expected = expected[expected.dataset == dataset].set_index(['condition', 'method'])
    actual = pd.DataFrame(rows).set_index(['condition', 'method'])
    assert set(expected.index) == set(actual.index), 'Missing or additional original baseline rows.'
    for key in ['accuracy', 'macro_f1', 'intervention_rate', 'helpful_rate', 'harmful_rate']:
        difference = (expected[key] - actual[key]).abs().max()
        assert difference < 1e-10, f'Baseline parity failed: {dataset}/{key}/{difference}'


def run_dataset(dataset):
    print('\n' + dataset.upper() + ' | loading existing action caches', flush=True)
    policy, validation = load_pack(dataset, 'policy'), load_pack(dataset, 'validation')
    assert not set(policy['subjects'].tolist()) & set(validation['subjects'].tolist()), 'Participant leakage.'
    assert not set(policy['window_ids'].tolist()) & set(validation['window_ids'].tolist()), 'Window leakage.'
    scale = np.maximum(np.quantile(policy['residuals'][0], 0.95, axis=0), 1e-8)
    rows_by_method, chosen = {}, baseline_choices(validation, scale)
    for method, actions in chosen.items():
        rows_by_method[method] = evaluate_choices(validation, actions, method)
    verify_baseline_parity(dataset, [row for rows in rows_by_method.values() for row in rows])
    print('  Original baseline metrics reproduced exactly.', flush=True)
    run = OUTPUT / f'{dataset}_fold_0_seed_{SEED}'
    run.mkdir(exist_ok=True)
    configuration = {
        'code_version': CODE_VERSION, 'cell_signature': CELL_SIGNATURE, 'dataset': dataset, 'seed': SEED,
        'max_iter': MAX_ITER, 'sklearn_version': sklearn.__version__, 'numpy_version': np.__version__,
        'model': {'learning_rate': 0.05, 'max_leaf_nodes': 15, 'max_depth': 4, 'min_samples_leaf': 40,
                  'l2_regularization': 5.0, 'max_bins': 64, 'early_stopping': False},
        'policy_inputs': policy['fingerprints'], 'validation_inputs': validation['fingerprints'],
        'empirical_selection_guards': {'clean_f1_tolerance': CLEAN_F1_TOLERANCE,
                                      'max_clean_harm': MAX_CLEAN_HARM, 'max_corrupted_harm': MAX_CORRUPTED_HARM},
        'harm_penalties': HARM_PENALTIES, 'utility_thresholds': UTILITY_THRESHOLDS,
        'confidence_margins': CONFIDENCE_MARGINS, 'residual_thresholds': RESIDUAL_THRESHOLDS
    }
    signature = json_signature(configuration)
    config_path = run / 'run_configuration.json'
    if config_path.exists():
        assert json.loads(config_path.read_text())['signature'] == signature, 'Inputs/settings changed; use a new OUTPUT folder.'
    save_json(config_path, {'signature': signature, 'configuration': configuration})
    X, feature_names = policy_features(policy['probabilities'], policy['residuals'], policy['statistics'], scale)
    fix, harm = outcome_targets(policy)
    print(f'  Policy fitting: {len(policy["y"])} original windows, {len(X)} action-case rows, {X.shape[1]} observable features.', flush=True)
    fix_model, fix_seconds = fit_head('benefit', X, fix, run, signature, feature_names)
    harm_model, harm_seconds = fit_head('harm', X, harm, run, signature, feature_names)
    target_counts = {'action_case_rows': len(X), 'unique_original_windows': len(policy['y']),
                     'beneficial': int(fix.sum()), 'harmful': int(harm.sum()),
                     'neutral': int(len(fix) - fix.sum() - harm.sum())}
    del X, fix, harm
    X, validation_feature_names = policy_features(validation['probabilities'], validation['residuals'], validation['statistics'], scale)
    assert validation_feature_names == feature_names
    started = time.perf_counter()
    shape = validation['probabilities'].shape[:2] + (validation['probabilities'].shape[2] - 1,)
    probability_fix = predict_head(fix_model, X).reshape(shape)
    probability_harm = predict_head(harm_model, X).reshape(shape)
    scoring_seconds = time.perf_counter() - started
    del X
    clean_keep_f1 = aggregate_rows(rows_by_method['keep'])['clean_macro_f1']
    selections, trial_rows = {}, []
    for family in ['harm_aware_policy', 'benefit_only_policy', 'confidence_margin_tuned', 'residual_mask_tuned', 'residual_reconstruct_tuned']:
        best, trials = select_settings(validation, probability_fix, probability_harm, scale, family, clean_keep_f1)
        chosen[family] = best['actions']
        rows_by_method[family] = evaluate_choices(validation, best['actions'], family)
        selections[family] = {'settings': best['settings'], 'validation_summary': best['summary'],
                              'keep_only_on_validation': bool(np.all(best['actions'] == 0))}
        trial_rows.extend({'dataset': dataset, **row} for row in trials)
    rows = [{'dataset': dataset, **row} for method_rows in rows_by_method.values() for row in method_rows]
    metrics = [{'dataset': dataset, 'method': method, **aggregate_rows(method_rows)} for method, method_rows in rows_by_method.items()]
    save_json(run / 'selected_policy.json', {'signature': signature, 'residual_scale': scale.tolist(),
              'feature_names': feature_names, 'action_names': validation['action_names'].tolist(),
              'selected_methods': selections, 'selection_split': 'validation',
              'note': 'Empirical validation constraints; no guaranteed deployment or shifted-data risk bound.'})
    save_json(run / 'feature_audit.json', {'allowed_arrays': ['probabilities', 'reconstruction_residuals', 'observed_group_statistics', 'candidate_action_identity'],
              'excluded_features': ['y', 'subjects', 'window_ids', 'source_indices', 'condition', 'severity', 'injected_fault_location'],
              'policy_validation_participant_overlap': 0, 'policy_validation_window_overlap': 0,
              'internal_random_row_early_stopping': False, 'feature_names': feature_names,
              'policy_outcome_counts': target_counts})
    methods = list(chosen)
    predictions = validation['probabilities'].argmax(axis=-1)
    selected_predictions = np.stack([np.take_along_axis(predictions, chosen[m][..., None], axis=2).squeeze(2) for m in methods])
    save_npz(run / 'validation_predictions.npz', methods=np.asarray(methods), conditions=np.asarray(validation['conditions']),
             choices=np.stack([chosen[m] for m in methods]), predictions=selected_predictions,
             y=validation['y'], subjects=validation['subjects'], window_ids=validation['window_ids'],
             probability_fix=probability_fix, probability_harm=probability_harm,
             signature=np.asarray(signature))
    costs = {'dataset': dataset, 'benefit_head_training_seconds': fix_seconds,
             'harm_head_training_seconds': harm_seconds, 'validation_head_scoring_seconds': scoring_seconds,
             'timing_excludes_candidate_generation': True}
    summary_frame = pd.DataFrame(metrics)
    summary_frame.to_csv(run / 'pilot_summary.csv', index=False)
    pd.DataFrame(rows).to_csv(run / 'validation_comparison.csv', index=False)
    save_json(run / 'dataset_complete.json', {'signature': signature, 'source_cache_signatures_verified': True,
              'original_baseline_parity': True, 'policy_outcome_counts': target_counts,
              'selected_methods': selections, 'test_predictions_computed': 0})
    print('  Selected policy:', selections['harm_aware_policy']['settings'],
          '| keep-only:', selections['harm_aware_policy']['keep_only_on_validation'], flush=True)
    del policy, validation, fix_model, harm_model
    gc.collect()
    return rows, metrics, trial_rows, costs, selections


def main():
    rows, metrics, trials, costs, selected = [], [], [], [], {}
    for dataset in ['dsads', 'har']:
        r, m, t, c, s = run_dataset(dataset)
        rows.extend(r)
        metrics.extend(m)
        trials.extend(t)
        costs.append(c)
        selected[dataset] = s
        save_csv(OUTPUT / 'notebook03_pilot_summary.csv', pd.DataFrame(metrics))
        save_csv(OUTPUT / 'notebook03_validation_comparison.csv', pd.DataFrame(rows))
        save_csv(OUTPUT / 'notebook03_selection_trials.csv', pd.DataFrame(trials))
    frame = pd.DataFrame(metrics)
    findings = {}
    for dataset in ['dsads', 'har']:
        part = frame[frame.dataset == dataset]
        learner = part[part.method == 'harm_aware_policy'].iloc[0]
        baselines = part[~part.method.isin(['harm_aware_policy', 'benefit_only_policy', 'oracle_diagnostic'])]
        baseline = baselines.loc[baselines.macro_f1.idxmax()]
        findings[dataset] = {'strongest_baseline_by_macro_f1': str(baseline['method']),
            'policy_minus_strongest_baseline_macro_f1_pp': float(100.0 * (learner['macro_f1'] - baseline['macro_f1'])),
            'validation_improvement_observed': bool(learner['macro_f1'] > baseline['macro_f1'] + 1e-12),
            'policy_keep_only': selected[dataset]['harm_aware_policy']['keep_only_on_validation']}
    report = {'code_version': CODE_VERSION, 'datasets_completed': ['dsads', 'har'], 'fold': 'fold_0', 'seed': SEED,
              'models_fit_on': 'policy partition only', 'threshold_selection_split': 'validation',
              'evaluation_split': 'same validation set used for threshold selection; exploratory development results',
              'original_baseline_parity': True, 'test_predictions_computed': 0, 'held_out_faults_evaluated': [],
              'learned_policy_trained': True, 'aggregation': 'equal-weight mean over nine corrupted conditions',
              'empirical_guards_are_not_risk_guarantees': True, 'bootstrap_or_significance_claims': False,
              'findings': findings, 'costs': costs, 'output_directory': str(OUTPUT)}
    save_json(OUTPUT / 'notebook03_run_report.json', report)
    save_json(OUTPUT / 'notebook03_selected_policies.json', selected)
    save_csv(OUTPUT / 'notebook03_training_cost.csv', pd.DataFrame(costs))
    inventory = [{'filename': str(p.relative_to(OUTPUT)), 'bytes': p.stat().st_size, 'sha256': file_sha256(p)}
                 for p in sorted(OUTPUT.rglob('*')) if p.is_file() and not p.name.endswith('.partial') and p.name != 'notebook03_artifact_inventory.json']
    save_json(OUTPUT / 'notebook03_artifact_inventory.json', inventory)
    print('\nNOTEBOOK 03 COMPLETE')
    print('VALIDATION SELECTION RESULTS | percentages | exploratory; final test reserved')
    display = frame[~frame.method.isin(['mask_suspect', 'reconstruct_suspect'])][
        ['dataset', 'method', 'clean_macro_f1', 'macro_f1', 'accuracy', 'intervention_rate', 'helpful_rate', 'harmful_rate']].copy()
    display.iloc[:, 2:] *= 100.0
    print(display.to_string(index=False, float_format=lambda v: f'{v:.2f}'))
    for dataset, finding in findings.items():
        print(f'{dataset.upper()}: learned policy minus strongest baseline Macro-F1 = '
              f'{finding["policy_minus_strongest_baseline_macro_f1_pp"]:+.2f} pp; '
              f'baseline={finding["strongest_baseline_by_macro_f1"]}; keep-only={finding["policy_keep_only"]}')
    print('Final test predictions computed: 0 | Held-out stuck faults evaluated: 0')
    print('Outputs:', OUTPUT)
    print('Share this final output and notebook03_pilot_summary.csv, notebook03_run_report.json, notebook03_selected_policies.json.')


try:
    with threadpool_limits(limits=CPU_THREADS):
        main()
except KeyboardInterrupt:
    print('\nNOTEBOOK 03 PAUSED. Completed heads are saved. Rerun this same cell to resume.')


Mounted at /content/drive
NOTEBOOK 03 | learned benefit-and-harm policy | CPU | one executable cell
Scope: both datasets, fold_0, seed 2026; validation selection only.

DSADS | loading existing action caches
  Original baseline metrics reproduced exactly.
  Policy fitting: 1140 original windows, 114000 action-case rows, 107 observable features.
  benefit head fitted | positive outcomes 2431/114000 | 10.9s
  harm head fitted | positive outcomes 5719/114000 | 10.1s
  Selected policy: {'family': 'harm_aware_policy', 'harm_penalty': 2.0, 'threshold': 0.02} | keep-only: False

HAR | loading existing action caches
  Original baseline metrics reproduced exactly.
  Policy fitting: 989 original windows, 39560 action-case rows, 50 observable features.
  benefit head fitted | positive outcomes 1324/39560 | 2.8s
  harm head fitted | positive outcomes 8500/39560 | 1.9s
  Selected policy: {'family': 'harm_aware_policy', 'harm_penalty': 1.0, 'threshold': 2.0} | keep-only: True

NOTEBOOK 03 COMPLETE
V